In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# ================================
# 1) Install libraries
# ================================
!pip install geopandas folium mapclassify

# ================================
# 2) Download Thailand shapefile
# ================================
!wget https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_THA_shp.zip
!unzip gadm41_THA_shp.zip

--2026-03-26 04:41:23--  https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_THA_shp.zip
Resolving geodata.ucdavis.edu (geodata.ucdavis.edu)... 128.120.146.30
Connecting to geodata.ucdavis.edu (geodata.ucdavis.edu)|128.120.146.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59691451 (57M) [application/zip]
Saving to: ‘gadm41_THA_shp.zip.1’

gadm41_THA_shp.zip. 100%[===================>]  56.93M  43.7MB/s    in 1.3s    

2026-03-26 04:41:25 (43.7 MB/s) - ‘gadm41_THA_shp.zip.1’ saved [59691451/59691451]

Archive:  gadm41_THA_shp.zip
replace gadm41_THA_0.cpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [27]:
# ================================
# 3) Import libraries
# ================================
import geopandas as gpd
import pandas as pd
import folium
import numpy as np # Import numpy for linspace

# ================================
# 4) Load shapefile (Province Level)
# ================================
gdf = gpd.read_file('gadm41_THA_1.shp')

# ================================
# 5) Create 77-province dataset
# ================================
data = {                       # similar to .csv or .dbf but mannual version
    'province': [
        'Bangkok Metropolis','Samut Prakan','Nonthaburi','Pathum Thani','Phra Nakhon Si Ayutthaya',
        'Ang Thong','Lop Buri','Sing Buri','Chai Nat','Saraburi','Chon Buri','Rayong','Chanthaburi',
        'Trat','Chachoengsao','Prachin Buri','Nakhon Nayok','Sa Kaeo','Nakhon Ratchasima','Buri Ram',
        'Surin','Si Sa Ket','Ubon Ratchathani','Yasothon','Chaiyaphum','Amnat Charoen','Bueng Kan',
        'Nong Bua Lamphu','Khon Kaen','Udon Thani','Loei','Nong Khai','Maha Sarakham','Roi Et',
        'Kalasin','Sakon Nakhon','Nakhon Phanom','Mukdahan','Chiang Mai','Lamphun','Lampang',
        'Uttaradit','Phrae','Nan','Phayao','Chiang Rai','Mae Hong Son','Nakhon Sawan','Uthai Thani',
        'Kamphaeng Phet','Tak','Sukhothai','Phitsanulok','Phichit','Phetchabun','Ratchaburi',
        'Kanchanaburi','Suphan Buri','Nakhon Pathom','Samut Sakhon','Samut Songkhram','Phetchaburi',
        'Prachuap Khiri Khan','Nakhon Si Thammarat','Krabi','Phang Nga','Phuket','Surat Thani',
        'Ranong','Chumphon','Songkhla','Satun','Trang','Phatthalung','Pattani','Yala','Narathiwat'
    ],
    'population': [
        5666264,1300000,1250000,1130020,820000,280000,760000,210000,330000,650000,
        1609800,720000,520000,230000,720000,490000,260000,530000,2633013,1550000,
        1400000,1500000,1850000,540000,1130000,380000,420000,510000,1791327,1586023,
        640000,520000,960000,1300000,980000,950000,720000,350000,1774063,530000,
        780000,460000,450000,480000,590000,128000,1040000,330000,720000,530000,
        780000,900000,540000,990000,850000,620000,780000,730000,230000,390000,
        480000,700000,600000,920000,410000,270000,418402,1068345,190000,510000,
        1400000,520000,630000,510000,730000,540000,820000
    ],
    'area_km2': [
        1568.7,1004.1,622.3,1525.9,2556.6,968.4,6196.5,822.5,2469.7,3576.5,
        4363.0,3552.0,6338.0,2819.0,5351.0,4762.0,2122.0,7195.0,20494.0,10321.0,
        8124.0,8839.0,16112.0,4100.0,12778.0,5161.0,4305.0,3859.0,10886.0,11730.0,
        11424.0,3736.0,5291.0,8299.0,6946.0,9605.0,5513.0,3395.0,20107.0,4505.0,
        12533.0,7838.0,6558.0,11472.0,6318.0,117678.0,9597.0,6730.0,10775.0,16406.0,
        6596.0,10549.0,4522.0,12668.0,5387.0,19685.0,5397.0,19483.0,5325.0,2168.0,
        416.9,622.0,535.0,16439.0,4708.5,8089.0,4170.9,543.0,12892.0,3298.0,
        5996.0,7393.0,2494.0,4942.0,3521.0,1940.0,4521.0
    ]
}

df = pd.DataFrame(data)

# ================================
# 6) Merge shapefile + table
# ================================
merged = gdf.merge(
    df,
    left_on='NAME_1',
    right_on='province',
    how='left'
)

# ================================
# 7) Calculate population density
# ================================
merged['density'] = merged['population'] / merged['area_km2']

# ================================
# 9) Interactive map (Folium)
# ================================
m = folium.Map(location=[13.7,100.5], zoom_start=6)

# Define even more bins for density using more quantiles to show finer frequency colors
density_bins = list(merged['density'].dropna().quantile(np.linspace(0, 1, 21))) # 20 quantiles

folium.Choropleth(
    geo_data=merged.to_json(),
    data=merged,
    columns=['NAME_1','density'],
    key_on='feature.properties.NAME_1',
    fill_color='YlOrRd',
    fill_opacity=0.9,
    line_opacity=0.8,
    legend_name='Population Density (people / km²)',
    threshold_scale=density_bins # Use the refined bins for more frequency colors
).add_to(m)

folium.GeoJson(
    merged,
    tooltip=folium.GeoJsonTooltip(
        fields=['NAME_1','population','area_km2','density'],
        aliases=['Province','Population','Area (km²)','Density'],
        localize=True
    )
).add_to(m)

m

m.save("thailand_density_map1.html")

In [28]:
# ================================
# 1) Import libraries
# ================================
import geopandas as gpd
import pandas as pd
import folium
import numpy as np
import branca.colormap as cm

# ================================
# 2) Load shapefile
# ================================
gdf = gpd.read_file('gadm41_THA_1.shp')

# ================================
# 3) Your dataset
# ================================
df = pd.DataFrame(data)

# ================================
# 4) Merge data
# ================================
merged = gdf.merge(
    df,
    left_on='NAME_1',
    right_on='province',
    how='left'
)

# ================================
# 5) Calculate density
# ================================
merged['density'] = merged['population'] / merged['area_km2']

# 🔥 IMPORTANT: Fix skew (Bangkok problem)
merged['density_log'] = np.log1p(merged['density'])

# ================================
# 6) Create base map
# ================================
m = folium.Map(
    location=[13.7, 100.5],
    zoom_start=6,
    tiles='CartoDB positron'
)

# ================================
# 7) Beautiful color scale (FULL RANGE 🌈)
# ================================
colormap = cm.LinearColormap(
    colors=[
        'navy',     # very low
        'blue',
        'cyan',
        'green',
        'yellow',
        'orange',
        'red'       # very high
    ],
    vmin=merged['density_log'].min(),
    vmax=merged['density_log'].max()
)

colormap.caption = 'Population Density (log scale)'

# ================================
# 8) Style function
# ================================
def style_function(feature):
    value = feature['properties']['density_log']
    return {
        'fillColor': colormap(value) if value else 'lightgray',
        'color': 'black',
        'weight': 0.6,
        'fillOpacity': 0.85
    }

# ================================
# 9) Highlight effect (hover 🔥)
# ================================
highlight_function = lambda x: {
    'weight': 3,
    'color': 'white',
    'fillOpacity': 1
}

# ================================
# 10) Tooltip (clean UI)
# ================================
tooltip = folium.GeoJsonTooltip(
    fields=['NAME_1','population','area_km2','density'],
    aliases=['Province:', 'Population:', 'Area (km²):', 'Density:'],
    localize=True,
    style="""
        background-color: white;
        color: black;
        font-family: Arial;
        font-size: 12px;
        padding: 10px;
        border-radius: 6px;
        box-shadow: 3px;
    """
)

# ================================
# 11) Add GeoJson layer
# ================================
folium.GeoJson(
    merged,
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=tooltip
).add_to(m)

# ================================
# 12) Add legend
# ================================
colormap.add_to(m)

# ================================
# 13) Add title (HTML overlay 🔥)
# ================================
title_html = '''
     <h3 align="center" style="font-size:20px">
     <b>Thailand Population Density Map</b><br>
     <small>(Log-scaled for better visualization)</small>
     </h3>
     '''
m.get_root().html.add_child(folium.Element(title_html))

# ================================
# 14) Layer control
# ================================
folium.LayerControl().add_to(m)

# ================================
# 15) Show map
# ================================
m

m.save("thailand_density_map.html")

credit_html = '''
<div style="
    position: fixed;
    bottom: 10px;
    left: 10px;
    z-index: 9999;
    background-color: white;
    padding: 8px 12px;
    border-radius: 6px;
    font-size: 12px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
">
<b>Created by Patimakorn</b><br>
Data: Thailand Provinces (GADM) + Population Dataset<br>
Tool: Python (GeoPandas, Folium)
</div>
'''
m.get_root().html.add_child(folium.Element(credit_html))

m.save("index.html")